In [1]:
# Project: Optimizing Hospitality - Hotel Data Scraping
# Purpose: Collect hotel name, price, rating, and review data
# Source: Booking.com
# Tools Used: Selenium, BeautifulSoup, Pandas


# Import required libraries

import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager

# List of Tier-1 cities to scrape hotel data

cities = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "Hyderabad",
    "Pune",
    "Ahmedabad",
    "Chennai",
    "Kolkata"
]

CHECKIN = "2026-02-01"
CHECKOUT = "2026-02-20"

MAX_PAGES = 5      # each page ≈ 25 hotels
SCROLL_COUNT = 10

BASE_URL = (
    "https://www.booking.com/searchresults.en-gb.html"
    "?ss={city}&checkin={checkin}&checkout={checkout}"
    "&group_adults=2&no_rooms=1&offset={offset}"
)

# -----------------------
# START CHROME (STEALTH)
# -----------------------
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Setup Chrome WebDriver for automated browser control

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# Remove webdriver flag

driver.execute_cdp_cmd(
    "Page.addScriptToEvaluateOnNewDocument",
    {
        "source": """
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        })
        """
    }
)

# Create empty list to store extracted hotel data

results = []

# -----------------------
# HELPER FUNCTIONS
# -----------------------
def scroll_all():
    for _ in range(SCROLL_COUNT):
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(1)


def extract_number(text):
    if not text:
        return None
    numbers = re.sub(r"[^\d]", "", text)
    return int(numbers) if numbers else None


# -----------------------
# SCRAPING LOGIC
# -----------------------
for city in cities:
    print(f"\n===== CITY: {city} =====")

    for page in range(MAX_PAGES):
        offset = page * 25
        url = BASE_URL.format(
            city=city,
            checkin=CHECKIN,
            checkout=CHECKOUT,
            offset=offset
        )

        print(f"Scraping page {page + 1}")
        driver.get(url)   # Open hotel search page for current city
        time.sleep(6)

        scroll_all()
        
# Parse webpage HTML using BeautifulSoup

        soup = BeautifulSoup(driver.page_source, "lxml")
        hotels = soup.select("div[data-testid='property-card']")

        print(f"Hotels found: {len(hotels)}")

        if not hotels:
            print("No more hotels. Moving to next city.")
            break

        for hotel in hotels:
            try:
                # Hotel Name
                name_elem = hotel.select_one("div[data-testid='title']")
                hotel_name = name_elem.get_text(strip=True) if name_elem else None

                # Address
                address_elem = hotel.select_one("span[data-testid='address']")
                address = address_elem.get_text(strip=True) if address_elem else None

                # Price
                price_elem = hotel.select_one("span[data-testid='price-and-discounted-price']")
                price_text = price_elem.get_text(strip=True) if price_elem else None
                price_numeric = extract_number(price_text)

                # Rating & Reviews
                rating_elem = hotel.select_one("div[data-testid='review-score']")
                rating_text = rating_elem.get_text(" ", strip=True) if rating_elem else None
                review_count = extract_number(rating_text)

                # Hotel URL
                link_elem = hotel.select_one("a[data-testid='title-link']")
                hotel_url = (
                    "https://www.booking.com" + link_elem["href"].split("?")[0]
                    if link_elem else None
                )

                results.append({
                    "City": city,
                    "Hotel Name": hotel_name,
                    "Address": address,
                    "Price Text": price_text,
                    "Price Numeric": price_numeric,
                    "Rating & Reviews": rating_text,
                    "Review Count": review_count,
                    "Hotel URL": hotel_url
                })

            except Exception as e:
                print("Error:", e)

# -----------------------
# SAVE FILES
# -----------------------

# Convert collected hotel data into pandas DataFrame

df = pd.DataFrame(results)

df.to_csv("booking_hotels_data.csv", index=False)
df.to_excel("booking_hotels_data.xlsx", index=False)

print("\n✅ Files saved successfully:")
print("✔ booking_hotels_data.csv")
print("✔ booking_hotels_data.xlsx")

driver.quit()


===== CITY: Mumbai =====
Scraping page 1
Hotels found: 25
Scraping page 2
Hotels found: 75
Scraping page 3
Hotels found: 75
Scraping page 4
Hotels found: 75
Scraping page 5
Hotels found: 75

===== CITY: Delhi =====
Scraping page 1
Hotels found: 75
Scraping page 2
Hotels found: 75
Scraping page 3
Hotels found: 75
Scraping page 4
Hotels found: 74
Scraping page 5
Hotels found: 75

===== CITY: Bangalore =====
Scraping page 1
Hotels found: 75
Scraping page 2
Hotels found: 75
Scraping page 3
Hotels found: 75
Scraping page 4
Hotels found: 76
Scraping page 5
Hotels found: 76

===== CITY: Hyderabad =====
Scraping page 1
Hotels found: 76
Scraping page 2
Hotels found: 76
Scraping page 3
Hotels found: 76
Scraping page 4
Hotels found: 76
Scraping page 5
Hotels found: 76

===== CITY: Pune =====
Scraping page 1
Hotels found: 75
Scraping page 2
Hotels found: 74
Scraping page 3
Hotels found: 73
Scraping page 4
Hotels found: 73
Scraping page 5
Hotels found: 73

===== CITY: Ahmedabad =====
Scraping page